# PMC Miner — Smoke Test

Two ways of mining the paper data through PMC:

1. Search-based mining: test top 5 hits.
2. DOI-based mining: DOIs from `notebook/test_dois.csv`.

In [1]:
from pathlib import Path
import json

from pmc_miner import SearchBasedMiner, DOIBasedMiner
from pmc_miner.utils.logging import setup_logging

setup_logging()

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
print("notebook dir :", NOTEBOOK_DIR)
print("project root :", PROJECT_ROOT)

notebook dir : /Users/yaochenr/project/pmc_data_mining/notebook
project root : /Users/yaochenr/project/pmc_data_mining


## 1. Search-based mining

Search PMC with keywords and mine the top 5 hits. You can specify the initial search keywords and post-filter keywords in: /Users/yaochenr/project/pmc_data_mining/pmc_miner/core/config.py.

It will firstly do a boarder search and then narrow dowm by using the filter.

By specifying `batch_size=5`, it will cap how many of those hits actually get downloaded + processed in this run (default would pull 100).

In [2]:
search_out = NOTEBOOK_DIR / "keyword_search_test"

search_miner = SearchBasedMiner(
    output_dir=str(search_out),
    keywords=["molecular glue degradation"],
)

search_stats = search_miner.mine_papers(max_papers=5, batch_size=5)
search_stats

2026-04-29 14:52:14,999 - INFO - Starting PMC search-based mining
2026-04-29 14:52:15,000 - INFO - Output dir: /Users/yaochenr/project/pmc_data_mining/notebook/keyword_search_test
2026-04-29 14:52:15,000 - INFO - Keywords: ['molecular glue degradation']
2026-04-29 14:52:16,069 - INFO - Found 5 papers in PMC Open Access subset
2026-04-29 14:52:16,610 - INFO - Retrieved metadata for 5 papers
2026-04-29 14:52:16,611 - INFO - Processing 5 papers (of 5 hits)
2026-04-29 14:52:16,611 - INFO - [1/5] PMC12341526
2026-04-29 14:52:17,461 - INFO - Successfully retrieved XML for PMC12341526
2026-04-29 14:52:17,514 - INFO - Saved metadata for PMCPMC12341526
2026-04-29 14:52:17,515 - INFO - Saved XML for PMCPMC12341526
2026-04-29 14:52:17,515 - INFO - Starting supplementary download for PMCPMC12341526
2026-04-29 14:52:17,516 - INFO - Starting supplementary download for PMC12341526
2026-04-29 14:52:18,589 - INFO - Downloading OA package from https://ftp.ncbi.nlm.nih.gov/pub/pmc/oa_package/a3/ba/PMC123

{'session_start_time': '2026-04-29T14:52:14.999749',
 'completion_time': '2026-04-29T14:52:33.491289',
 'search_keywords': ['molecular glue degradation'],
 'post_filter_keywords': ['degrader',
  'degradation',
  'proteasome',
  'TPD',
  'targeted protein degradation',
  'protein degradation'],
 'initial_papers_found': 5,
 'initial_papers_stored': 5,
 'filtered_papers_stored': 5,
 'failed_papers': 0,
 'filter_success_rate': 100.0,
 'storage_locations': {'init': '/Users/yaochenr/project/pmc_data_mining/notebook/keyword_search_test/init_papers',
  'filtered': '/Users/yaochenr/project/pmc_data_mining/notebook/keyword_search_test/filtered_papers'},
 'stored_papers': {'initial': ['PMC12341526',
   'PMC13001483',
   'PMC8685278',
   'PMC12491679',
   'PMC9428674'],
  'filtered': ['PMC12341526',
   'PMC13001483',
   'PMC8685278',
   'PMC12491679',
   'PMC9428674'],
  'failed': []}}

In [3]:
init_dir = search_out / "init_papers"
filtered_dir = search_out / "filtered_papers"

print("init_papers   :", sorted(p.name for p in init_dir.glob("PMC*")))
print("filtered_papers:", sorted(p.name for p in filtered_dir.glob("PMC*")))

init_papers   : ['PMC12341526', 'PMC12491679', 'PMC13001483', 'PMC8685278', 'PMC9428674']
filtered_papers: ['PMC12341526', 'PMC12491679', 'PMC13001483', 'PMC8685278', 'PMC9428674']


In [4]:
pmc_dirs = sorted(init_dir.glob("PMC*"))
if pmc_dirs:
    sample = pmc_dirs[0]
    meta = json.loads((sample / "metadata.json").read_text())
    processed = json.loads((sample / "processed.json").read_text())
    print(f"{sample.name}: {meta.get('title', '')[:100]}")
    print(f"  journal: {meta.get('journal')}")
    print(f"  doi    : {meta.get('doi')}")
    print(f"  sections : {len(processed.get('sections', []))}")
    print(f"  tables   : {len(processed.get('tables', []))}")
    print(f"  figures  : {len(processed.get('figures', []))}")
else:
    print("No papers stored.")

PMC12341526: Controlling CRISPR-Cas9 genome editing in human cells using a molecular glue degrader.
  journal: Molecular therapy. Nucleic acids
  doi    : 10.1016/j.omtn.2025.102640
  sections : 9
  tables   : 0
  figures  : 6


## 2. DOI-based mining

By using DOI-based mining, you only need to specify the DOI list. For papers not in PMC open access, they will be skipped and logged to `/dois_notin_pmc.txt`.

In [2]:
doi_csv = '/Users/yaochenr/project/pmc_data_mining/notebook/test_dois.csv'
doi_out = NOTEBOOK_DIR / "doi_mining_test"

doi_miner = DOIBasedMiner(
    output_dir=str(doi_out),
    paper_type="test", # a label
    download_images=True,
)

doi_stats = doi_miner.mine_from_csv(str(doi_csv))
doi_stats

2026-04-29 15:23:36,061 - INFO - Loaded 4 DOIs from /Users/yaochenr/project/pmc_data_mining/notebook/test_dois.csv
2026-04-29 15:23:36,061 - INFO - DOI-based mining (test): 4 DOIs
2026-04-29 15:23:36,062 - INFO - [1/4] 10.1021/acs.jmedchem.1c02175
2026-04-29 15:23:36,979 - INFO - Found PMC ID PMC9234961 for DOI 10.1021/acs.jmedchem.1c02175 (Open Access)
2026-04-29 15:23:37,520 - INFO - Retrieved metadata for 1 papers
2026-04-29 15:23:38,318 - INFO - Successfully retrieved XML for PMC9234961
2026-04-29 15:23:38,382 - INFO - Saved metadata for PMCPMC9234961
2026-04-29 15:23:38,383 - INFO - Saved XML for PMCPMC9234961
2026-04-29 15:23:38,383 - INFO - Starting supplementary download for PMCPMC9234961
2026-04-29 15:23:38,384 - INFO - Starting supplementary download for PMC9234961
2026-04-29 15:23:38,729 - INFO - Downloading OA package from https://ftp.ncbi.nlm.nih.gov/pub/pmc/oa_package/1c/8b/PMC9234961.tar.gz
2026-04-29 15:23:39,386 - ERROR - Failed to download package: HTTP 404
2026-04-29

{'session_start_time': '2026-04-29T15:23:36.061981',
 'completion_time': '2026-04-29T15:24:09.373398',
 'source_file': '/Users/yaochenr/project/pmc_data_mining/notebook/test_dois.csv',
 'output_directory': '/Users/yaochenr/project/pmc_data_mining/notebook/doi_mining_test',
 'paper_type': 'test',
 'statistics': {'total_dois': 4,
  'found_pmc_ids': 4,
  'successfully_processed': 4,
  'failed_processing': 0,
  'not_in_pmc': 0,
  'not_open_access': 0,
  'already_processed': 0},
 'successful_papers': [{'doi': '10.1021/acs.jmedchem.1c02175',
   'pmc_id': 'PMC9234961'},
  {'doi': '10.1126/science.adk4422', 'pmc_id': 'PMC11203266'},
  {'doi': '10.1016/j.ejmech.2024.116904', 'pmc_id': 'PMC11960843'},
  {'doi': '10.1038/s41467-025-58431-z', 'pmc_id': 'PMC12046021'}],
 'failed_papers': []}

In [3]:
pmc_dirs = sorted(doi_out.glob("PMC*"))
print(f"Stored {len(pmc_dirs)} papers:")
for p in pmc_dirs:
    meta = json.loads((p / "metadata.json").read_text())
    n_images = len(list((p / "images").glob("*.jpg"))) if (p / "images").exists() else 0
    n_supp = len(list((p / "supplementary").iterdir())) if (p / "supplementary").exists() else 0
    print(f"  {p.name}  doi={meta.get('doi')}  images={n_images}  supp_files={n_supp}")

skipped = doi_out / "dois_notin_pmc.txt"
if skipped.exists():
    print("\nSkipped (not in PMC OA):")
    print(skipped.read_text())

Stored 4 papers:
  PMC11203266  doi=10.1126/science.adk4422  images=5  supp_files=0
  PMC11960843  doi=10.1016/j.ejmech.2024.116904  images=4  supp_files=0
  PMC12046021  doi=10.1038/s41467-025-58431-z  images=7  supp_files=0
  PMC9234961  doi=10.1021/acs.jmedchem.1c02175  images=4  supp_files=0
